# Soft-skill answer classifier (Unit 4 - discriminative model)

We train a **discriminative** model that maps an answer to `poor / average / good`, in contrast to the **generative** LLM rubric evaluator that writes reasons. Both run side by side in the app.

**Honesty box.** The dataset is small and LLM-authored (see `data/DATASET_CARD.md`); answer *length* correlates strongly with the label. We therefore (1) report a length-only baseline, (2) evaluate on **unseen questions**, and (3) probe for sensitivity to names and colleges. High accuracy here does *not* mean the model can grade real interview answers.

In [ ]:
# --- Setup: works locally (from the repo) and on Colab (clone the repo first) -------------------------
import os, sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data" / "soft_skill_labeled.csv").exists():
    ROOT = ROOT.parent                       # notebook opened from notebooks/
if not (ROOT / "data" / "soft_skill_labeled.csv").exists():
    raise SystemExit("Open this notebook from the repository (or `git clone` it on Colab and cd into it).")
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
os.chdir(ROOT)
print("repo:", ROOT)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from backend.services import classifier_service as C
df = C.load_dataset()
print(df.shape); df.label.value_counts()

## 1. Look at the data - including the length confound

In [ ]:
df["words"] = df.answer.str.split().str.len()
df.boxplot(column="words", by="label", grid=False); plt.suptitle(""); plt.title("Answer length by label"); plt.ylabel("words"); plt.show()
df.groupby("label").words.describe()[["mean", "min", "max"]]

## 2. Models and evaluation protocols
`evaluate()` compares a length-only baseline, TF-IDF only, and TF-IDF + structural features under random CV and under leave-questions-out.

In [ ]:
ev = C.evaluate()
rows = [{"variant": k, "random_cv_acc": v["random_cv"]["accuracy_mean"], "leave_questions_out_acc": v["leave_questions_out"]["accuracy"],
         "leave_questions_out_f1": v["leave_questions_out"]["f1_macro"]} for k, v in ev["variants"].items()]
pd.DataFrame(rows).set_index("variant")

In [ ]:
cm = np.array(ev["variants"]["tfidf+features"]["leave_questions_out"]["confusion_matrix"])
ConfusionMatrixDisplay(cm, display_labels=["poor", "average", "good"]).plot(cmap="Blues"); plt.title("TF-IDF+features, leave-questions-out"); plt.show()
print(ev["note"])

## 3. What did it learn? Top TF-IDF terms per class

In [ ]:
pipe = C.build_pipeline("tfidf_only").fit(df.answer.tolist(), df.label)
vec, lr = pipe.named_steps["feats"], pipe.named_steps["lr"]   # tfidf_only variant: the vectorizer is the feature step
names = np.array(vec.get_feature_names_out())
for i, cls in enumerate(lr.classes_):
    print(cls.upper().ljust(8), ", ".join(names[np.argsort(lr.coef_[i])[-10:][::-1]]))

## 4. Fairness probe (counterfactual identity sentences)
We append a one-sentence *identity statement* of similar length to every answer and check whether predictions change. A grader that judges content should be insensitive to where you studied or your name. This is a **smoke test, not an audit**: it only covers a few phrases and one model.

In [ ]:
model = C.train(save=False)
base_pred = model.predict(df.answer.tolist()); base_p = model.predict_proba(df.answer.tolist())
probes = {
    "elite college":  "I studied at a well known institute in a big metro city.",
    "small college":  "I studied at a small local college in a village town.",
    "name A":         "My name is Priya Raghavan and I enjoy this kind of work.",
    "name B":         "My name is Mohammed Irfan and I enjoy this kind of work.",
}
rows = []
for name, sentence in probes.items():
    mod = (df.answer + " " + sentence).tolist()
    pred = model.predict(mod); proba = model.predict_proba(mod)
    good = list(model.classes_).index("good")
    rows.append({"appended sentence": name, "words added": len(sentence.split()), "predictions changed": int((pred != base_pred).sum()),
                 "mean shift in P(good)": round(float((proba[:, good] - base_p[:, good]).mean()), 4)})
pd.DataFrame(rows)

Compare the pairs: if `elite college` vs `small college` (or `name A` vs `name B`) shift P(good) differently, the model is reacting to identity rather than content. Differences caused only by the extra *length* affect both sentences similarly.

## 5. Error analysis

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
pred = cross_val_predict(C.build_pipeline("tfidf+features"), df.answer.tolist(), df.label, groups=df.question_id, cv=GroupKFold(4))
wrong = df[pred != df.label].assign(pred=pred[pred != df.label])
print(len(wrong), "errors out of", len(df))
for _, r in wrong.sample(min(4, len(wrong)), random_state=1).iterrows():
    print(f"\n[true={r.label} pred={r.pred}] {r.answer[:220]}")

## 6. Optional: DistilBERT
`python scripts/train_distilbert_classifier.py --epochs 4` fine-tunes DistilBERT on the same question-grouped split and saves it where the backend finds it automatically (shown next to the classical model in the UI). It needs `torch`/`transformers`; a GPU helps but is not required.

## 7. Discussion
- Why report a length-only baseline? What does it tell you about the dataset?
- Random CV vs leave-questions-out: why are the numbers different?
- What additional data would you collect, and who should label it, to make this classifier trustworthy?